In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs
from helpers.data_display import disp_pct_succ_on_mask
from helpers.check_model_performance import review_typical_slices, find_random_sample_given_mask
from process_dumb_models import run_against_models

In [3]:
ot_train, ot_train_label = load_pairs("overture", "train")
ot_val, ot_val_label = load_pairs("overture", "val")
ot_test, ot_test_label = load_pairs("overture", "test")

# House number + street remainder for the similarity features (helpers/pairs.py):
# adds number_a/b, street_a/b, number_missing, both_number_missing
ot_train, ot_val, ot_test = [add_number_street(df) for df in [ot_train, ot_val, ot_test]]

In [4]:
# Full feature build (cached; ~1.5 min on first run)
train_overture, val_overture, test_overture = get_processed_dfs([
    [ot_train, "./overture_training/overture_evaluation_train.parquet"],
    [ot_val, "./overture_training/overture_evaluation_val.parquet"],
    [ot_test, "./overture_training/overture_evaluation_test.parquet"],
])

In [5]:
# Overture -> Overture, scored on val
ret = run_against_models(train_overture, ot_train_label, val_overture, test_overture)

In [8]:
ret.keys()

dict_keys(['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier'])

In [7]:
for model in ['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier']:
    print(f"{'='*20} {model} {'='*20}")
    print(review_typical_slices(ot_val, ret[model]["val"][0], ot_val_label))
    print()


==================== LogisticRegression ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3389       1    100.0%
1                  positives: non-identical    797      652     145     81.8%
2     neg: case1 (same street, diff number)  13781    13753      28     99.8%
3  neg: case2 (same number, similar street)     34        1      33      2.9%
4                         neg: case3 (easy)  23863    23857       6    100.0%
5                        overall (accuracy)  41865    41652     213     99.5%

==================== RandomForestClassifier ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3390       0    100.0%
1                  positives: non-identical    797      681     116     85.4%
2     neg: case1 (same street, diff number)  13781    13715      66     99.5%
3  neg: case2 (

In [47]:
sample = find_random_sample_given_mask(ot_val, ret["LogisticRegression"]["val"][0], ot_val_label, 0)
print(sample)

==========positives: identical==========
                 address_a            address_b ZipCode_a ZipCode_b split  \
1120     227 COMMERCIAL ST    227 COMMERCIAL ST     06795     06795   val   
16138        510 N MAIN ST        510 N MAIN ST     06042     06042   val   
23299     27 GROVE HILL ST     27 GROVE HILL ST     06052     06052   val   
28229         382 AMITY RD         382 AMITY RD     06525     06525   val   
18105         470 RIVER RD         470 RIVER RD     06279     06279   val   
19031       124 SCANTIC RD       124 SCANTIC RD     06088     06088   val   
3023         158 EDISON RD        158 EDISON RD     06611     06611   val   
3141         5 COMMERCE DR        5 COMMERCE DR     06484     06484   val   
6281   360 CONNECTICUT AVE  360 CONNECTICUT AVE     06854     06854   val   
16788     239 GLENVILLE RD     239 GLENVILLE RD     06831     06831   val   

                                  gers_id_x  \
1120   f542bfb5-dab0-48d4-b666-e5e106a98003   
16138  a565c6c7-6

In [46]:
for model, splits in ret.items():
    preds, probs = splits["val"]
    # same_mask = (ot_val["address_a"] == ot_val["address_b"]) & (ot_val_label == 1)
    # disp_pct_succ_on_mask(ot_val_label, preds, same_mask, f"identical -- {model}",show_other = False )
    diff_mask = (ot_val["address_a"] != ot_val["address_b"]) & (ot_val_label == 1)
    disp_pct_succ_on_mask(ot_val_label, preds, diff_mask, f"diff -- {model}",show_other = False )
    


====================diff -- LogisticRegression====================
  Mask[Good](n=797) Mask[Bad](n=797)
0     0.82(n = 652)    0.18(n = 145)
====================diff -- RandomForestClassifier====================
  Mask[Good](n=797) Mask[Bad](n=797)
0     0.85(n = 681)    0.15(n = 116)
====================diff -- XGBClassifier====================
  Mask[Good](n=797) Mask[Bad](n=797)
0     0.85(n = 677)    0.15(n = 120)


### Results — Overture → Overture, val (2026-10-11)

23 features (22 similarities + `number_missing`), default hyperparameters, no class weighting.

| model | P (match) | R (match) | F1 (match) | case1 rejected | case2 rejected | case3 rejected |
|---|---|---|---|---|---|---|
| LogReg | 0.98 | 0.96 | 0.97 | 99.8% | 3% | 99.97% |
| RF | 0.97 | 0.97 | 0.97 | 99.5% | 18% | 99.9% |
| XGBoost | 0.97 | 0.97 | 0.97 | 99.5% | 21% | 99.9% |

~225 errors out of 41,865 val pairs per model. Read with care:
- **Headline likely inflated** by identical-string positives (~70% raw) → next: recall on identical vs non-identical positives.
- **Case 2** (directional variants, e.g. `N MAIN ST` / `S MAIN ST`) mostly accepted as matches — Step 1b's false-positive problem on real data; only 34 val rows.
- **Visible error sources:** text before the number on side B → `number_missing`; `ST`/`STREET` suffix variants.

Next: identical vs non-identical breakdown + errors by `number_missing`; Stamford → Overture; ZIP ablation.

*Diagnostic queries + results for this section: **Diagnostics** at the bottom of this notebook → Overture baseline.*

## Diagnostics

One-off checks kept out of the pipeline above. Paths are relative to this notebook's folder.

### House-number regex

**Problem:** The old `^(\d+)\s+` pattern missed numbers like `12A` and `16-18`, leaving them in the street name.  
**Check:** New regex on hand-picked cases; streets still starting with a digit; raw addresses the old pattern missed.  
**Result:** Hand cases correct; 473 / 492 streets start with a digit (ordinals, fractions, double numbers); old pattern missed 1,202 / 1,484 letter suffixes and 444 / 742 ranges (A / B) (re-run 2026-10-13)

```python
# Test the house-number token regex on hand-picked cases
check = pd.DataFrame({"address": ["12 MAIN ST", "12A MAIN ST", "16-18 ELM ST", "20-A ELM ST"]})
check["check"] = check["address"].str.extract(r"^(\d[A-Z0-9a-z-]*)\s")
print(check)

# After parsing: sample + count of streets that still start with a digit (ordinal streets, 1/2 addresses, double numbers)
ret = pd.concat([ot_train, ot_val, ot_test])
print(ret[ret["street_a"].str.match(r"^\d")][["street_a"]].sample(frac=1).head(10))
print(ret["street_a"].str.match(r"^\d").sum())
print(ret[ret["street_b"].str.match(r"^\d")][["street_b"]].sample(frac=1).head(10))
print(ret["street_b"].str.match(r"^\d").sum())

# Raw addresses the old ^(\d+)\s+ pattern missed: letter suffix (12A) on A / B, then ranges (16-18) on A / B
print(ret["address_a"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_b"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_a"].str.match(r"^[0-9]+-[0-9]+\s").sum())
print(ret["address_b"].str.match(r"^[0-9]+-[0-9]+\s").sum())
```

### Streets under 2 characters

**Problem:** The Overlap bigram metric divides by zero on strings shorter than 2 characters.  
**Check:** Find any short street names.  
**Result:** Test split only: street `1` (7 on A, 11 on B) → guarded in `features.py` (re-run 2026-10-13)

```python
# Streets under 2 characters (break bigram metrics): only the test addresses that are literally "1"
for df in [ot_train, ot_val, ot_test]:
    print(df[df["street_a"].str.len() < 2]["street_a"])
    print(df[df["street_b"].str.len() < 2]["street_b"])
    print()
```

### Positives with identical addresses

**Problem:** Suspected the models were misclassifying positives whose addresses are identical.  
**Check:** Misclassifications within `case0`, split by identical vs. different address and by number status.  
**Result:** Identical: LR 1 / 3,390 wrong (a ZIP disagreement), RF 0, XGB 0. Different: 82–85% correct. The `number_missing` subset is empty by construction since `both_number_missing` was split out; use `both_number_missing` instead (re-run 2026-10-13)

```python
print([k for k in ret.keys()])
mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] == ot_val["address_b"])).values
totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
print([int(c) for c in totals])

# missing = [ot_val[mask][(preds != ot_val_label) & (ot_val["number_missing"] == False)] for model, (preds, probs) in ret.items()]
# for miss,model in zip(missing,[k for k in ret.keys()]):
#     print(f"{'='*20} {model} {'='*20}")
#     print(miss[["address_a", "address_b", "ZipCode_a", "ZipCode_b"]])


# mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] != ot_val["address_b"])).values
# totals = [(preds[mask] == ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([int(c) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == True)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == False)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# print(
# len(ot_val[
#     (ot_val["neg_type"] == "case0") & 
#     (ot_val["address_a"] == ot_val["address_b"]) & 
#     (ot_val['number_missing'] == True)
# ]))
```